# Modelling 4: Explaining predictions and analysing errors
**Input:** the preprocessed features, plus the tuned settings and chosen model saved by notebook 1
(`tuned_hyperparameters.json` and `chosen_model.json`). This notebook answers three questions:
1. Are the test errors consistent across seeds?
2. What do the predictions rely on: the text, the image or the agreement signals?
3. Which kinds of posts does the chosen model get wrong, and why?

In [1]:
# Make the shared code in src/ importable, wherever the repository folder is
import os, sys
root = os.getcwd()
while not os.path.isdir(os.path.join(root, "src", "mvsa")) and root != os.path.dirname(root):
    root = os.path.dirname(root)
sys.path.insert(0, os.path.join(root, "src"))

from mvsa.modeling import *

CFG = json.load(open(os.path.join(OUT_DIR, "tuned_hyperparameters.json")))
chosen = json.load(open(os.path.join(OUT_DIR, "chosen_model.json")))
best_fam, best_setup = chosen["chosen_family"], chosen["chosen_setup"]
test, yte = D["test"], D["test"]["y"]
# Retrain every setup of the chosen family for each seed (seconds for logistic regression)
models = {seed: build(best_fam, CFG[best_fam], seed) for seed in SEEDS}
print(f"Chosen model: {best_fam} + {best_setup}; seeds {SEEDS}")

Loaded from <preprocessed folder>
  train  3591 posts | CLIP text 512-d, image 512-d
  val     446 posts | CLIP text 512-d, image 512-d
  test    447 posts | CLIP text 512-d, image 512-d
  hard    355 posts | CLIP text 512-d, image 512-d
Train class counts: {'negative': np.int64(1078), 'neutral': np.int64(376), 'positive': np.int64(2137)}


Chosen model: logreg + late; seeds [0, 1, 2, 3, 4]


## 1. Confusion matrices over the seeds
Rows are the true classes and columns the predictions, as mean ± standard deviation over the seeds. Logistic regression
is deterministic, so the chosen model's matrix is the same for every seed. Gated fusion varies, because its training
inputs come from cross-validation folds that change with the seed.

In [2]:
cms = {}
for setup in (best_setup, "gated"):
    mats = np.array([confusion_matrix(yte, models[s][0][setup](test).argmax(1), labels=[0, 1, 2]) for s in SEEDS])
    cms[setup] = mats
    mean, std = mats.mean(0), mats.std(0)
    table = pd.DataFrame([[f"{m:.1f} ± {s:.1f}" for m, s in zip(mr, sr)] for mr, sr in zip(mean, std)],
                         index=[f"true {l}" for l in LABELS], columns=[f"predicted {l}" for l in LABELS])
    print(f"{best_fam} + {setup}, test set, mean ± std over {len(SEEDS)} seeds:")
    print(table.to_string(), "\n")
pd.concat({s: pd.DataFrame(m.mean(0), index=LABELS, columns=LABELS) for s, m in cms.items()}).to_csv(
    os.path.join(OUT_DIR, "confusion_matrices_mean_over_seeds.csv"))

logreg + late, test set, mean ± std over 5 seeds:
              predicted negative predicted neutral predicted positive
true negative         94.0 ± 0.0        14.0 ± 0.0         26.0 ± 0.0
true neutral          10.0 ± 0.0        19.0 ± 0.0         10.0 ± 0.0
true positive         40.0 ± 0.0        36.0 ± 0.0        198.0 ± 0.0 

logreg + gated, test set, mean ± std over 5 seeds:
              predicted negative predicted neutral predicted positive
true negative         89.6 ± 2.0        22.2 ± 1.2         22.2 ± 1.0
true neutral           8.0 ± 0.6        24.0 ± 0.6          7.0 ± 0.0
true positive         37.0 ± 0.6        49.2 ± 1.9        187.8 ± 1.5 



## 2. What the predictions rely on: permutation importance
Permutation importance shuffles one input across the test posts and measures how much test macro-F1 drops. A large
drop means the model relies on that input. Shuffling breaks the link between the input and the label but keeps the
input's distribution, so it is a fairer test than setting the input to zero. Each input is shuffled 30 times.

**2a. Text versus image.** Each modality's whole CLIP embedding is shuffled at once, for early, late and gated fusion.

In [3]:
def macro_f1_of(predict, d):
    return f1_score(d["y"], predict(d).argmax(1), average="macro")

def modality_importance(predict, d, repeats=30, seed=0):
    """Mean and std of the drop in macro-F1 when one modality's embedding is shuffled across posts."""
    rng = np.random.default_rng(seed)
    base, out = macro_f1_of(predict, d), {}
    for modality in ("text", "image"):
        drops = []
        for _ in range(repeats):
            shuffled = dict(d)
            shuffled[modality] = d[modality][rng.permutation(len(d["y"]))]
            drops.append(base - macro_f1_of(predict, shuffled))
        out[modality] = (np.mean(drops), np.std(drops))
    return base, out

rows = []
for setup in ("early", "late", "gated"):
    base, imp = modality_importance(models[SEEDS[0]][0][setup], test)
    for modality, (m, s) in imp.items():
        rows.append({"setup": setup, "shuffled": modality, "macro_f1": base, "drop_mean": m, "drop_std": s})
modality_imp = pd.DataFrame(rows)
print(f"Drop in test macro-F1 when one modality is shuffled ({best_fam}, seed {SEEDS[0]}):")
print(modality_imp.round(3).to_string(index=False))
modality_imp.to_csv(os.path.join(OUT_DIR, "permutation_importance_modality.csv"), index=False)

Drop in test macro-F1 when one modality is shuffled (logreg, seed 0):
setup shuffled  macro_f1  drop_mean  drop_std
early     text     0.609      0.123     0.019
early    image     0.609      0.155     0.025
 late     text     0.603      0.094     0.017
 late    image     0.603      0.124     0.020
gated     text     0.602      0.092     0.018
gated    image     0.602      0.146     0.024


**2b. Inside the gate.** The gate's ten inputs are shuffled in four groups: the text model's probabilities, the image
model's probabilities, the two agreement signals (CLIP cosine similarity and Jensen-Shannon divergence), and the two
confidence values. This is repeated for each seed's gate.

In [4]:
GROUPS = {"text probabilities": [0, 1, 2], "image probabilities": [3, 4, 5],
          "agreement (cosine, JS divergence)": [6, 7], "confidence (text, image)": [8, 9]}
rows = []
for seed in SEEDS:
    predictors, gate = models[seed]
    G = gate_inputs(predictors["text_only"](test), predictors["image_only"](test), test)
    base = f1_score(yte, gate.predict(G), average="macro")
    rng = np.random.default_rng(seed)
    for name, cols in GROUPS.items():
        for _ in range(30):
            shuffled = G.copy()
            shuffled[:, cols] = G[rng.permutation(len(G))][:, cols]   # the group's columns move together
            rows.append({"seed": seed, "group": name, "drop": base - f1_score(yte, gate.predict(shuffled), average="macro")})
gate_imp = pd.DataFrame(rows).groupby("group")["drop"].agg(["mean", "std"]).reindex(list(GROUPS))
print(f"Drop in the gate's test macro-F1 when a group is shuffled ({best_fam}, {len(SEEDS)} seeds x 30 shuffles):")
print(gate_imp.round(3).to_string())
gate_imp.to_csv(os.path.join(OUT_DIR, "permutation_importance_gate.csv"))

Drop in the gate's test macro-F1 when a group is shuffled (logreg, 5 seeds x 30 shuffles):
                                    mean    std
group                                          
text probabilities                 0.085  0.020
image probabilities                0.156  0.022
agreement (cosine, JS divergence)  0.009  0.010
confidence (text, image)          -0.001  0.007


## 3. Error analysis of the chosen model
The first table counts each kind of mistake. The second compares error rates for groups of posts: whether the
separate text and image models agree, how confident the chosen model is, and whether the two human labels agreed.

In [5]:
predictors = models[SEEDS[0]][0]
P = predictors[best_setup](test)
pred, conf = P.argmax(1), P.max(1)
text_pred, image_pred = predictors["text_only"](test).argmax(1), predictors["image_only"](test).argmax(1)
wrong = pred != yte
errors = pd.DataFrame({"true": np.array(LABELS)[yte], "predicted": np.array(LABELS)[pred], "score": conf,
                       "text model": np.array(LABELS)[text_pred], "image model": np.array(LABELS)[image_pred],
                       "caption": posts.loc[test["ids"], "text_clean"].to_numpy()})[wrong]
print(f"{wrong.sum()} of {len(yte)} test posts misclassified ({wrong.mean():.1%})\n")
print("Most common mistakes:")
print(errors.groupby(["true", "predicted"]).size().sort_values(ascending=False).rename("posts").to_string(), "\n")

agree = text_pred == image_pred
groups = {"all test posts": np.ones_like(wrong), "text and image models agree": agree,
          "text and image models disagree": ~agree, "score 0.70 or more": conf >= 0.70,
          "score below 0.45": conf < 0.45, "human labels agree": test["group"] == "agreement",
          "soft mismatch": test["group"] == "soft_mismatch"}
error_rates = pd.DataFrame({"posts": [m.sum() for m in groups.values()],
                            "error rate": [wrong[m].mean() for m in groups.values()]}, index=list(groups))
print(error_rates.round(3).to_string())
error_rates.to_csv(os.path.join(OUT_DIR, "error_rates.csv"))

136 of 447 test posts misclassified (30.4%)

Most common mistakes:
true      predicted
positive  negative     40
          neutral      36
negative  positive     26
          neutral      14
neutral   negative     10
          positive     10 

                                posts  error rate
all test posts                    447       0.304
text and image models agree       246       0.199
text and image models disagree    201       0.433
score 0.70 or more                166       0.127
score below 0.45                   39       0.667
human labels agree                253       0.241
soft mismatch                     194       0.387


### The most confident mistakes
For each common kind of mistake, these are the posts the model got wrong with the highest score. Reading them shows
what the model misunderstands. Mentions are already replaced by `@user` in the cleaned text.

In [6]:
pd.set_option("display.max_colwidth", 70)
for true, predicted in [("neutral", "positive"), ("negative", "positive"), ("positive", "negative"), ("positive", "neutral")]:
    match = errors[(errors["true"] == true) & (errors["predicted"] == predicted)]
    print(f"\nTrue {true}, predicted {predicted} ({len(match)} posts):")
    print(match.sort_values("score", ascending=False).head(4)[["score", "text model", "image model", "caption"]]
          .round(2).to_string(index=False))


True neutral, predicted positive (10 posts):
 score text model image model                                                                                    caption
  0.61   positive    positive           Vintage Antique Eugene DeLacroix French Art Print Horse Frightened by Storm 1824
  0.60   positive    positive                                               Why are you feeling downcast? Take the quiz:
  0.56   positive    positive Im liking @user Shoes ?? prefers to be known as an enthusiastic than an expert @user wpfut
  0.52   positive    negative   tiny guy on A5..inspired by a recent doodle I did for @user (wihch I neglected to scan!)

True negative, predicted positive (26 posts):
 score text model image model                                                                                                             caption
  0.88   positive    positive Really getting peed off with the dismal summer weather here in Edinburgh. Oh to be back here summer cruising monaco
  0.86   p

### Hard-mismatch posts
These posts have a positive label for one modality and a negative label for the other, so there is no single right
answer. The table shows which human label the chosen model's prediction matches.

In [7]:
hard = D["hard"]
labels = np.array(LABELS)[predictors[best_setup](hard).argmax(1)]
follows = np.where(labels == hard["text_label"], "text label",
                   np.where(labels == hard["image_label"], "image label", "neither (neutral)"))
print(pd.Series(follows).value_counts(normalize=True).round(3).to_string())

image label          0.510
text label           0.411
neither (neutral)    0.079
